### Building a self-correcting multi-step reasoning system using LangChain + AzureChatOpenAI

In [ ]:
#Mimicing how human thinks
"""
User Question
   ↓
Step 1: Plan (break problem down)
   ↓
Step 2: Solve (initial answer)
   ↓
Step 3: Critique (find mistakes)
   ↓
Step 4: Improve (fix answer)
   ↓
"""

In [1]:
from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
import os
import dotenv
from dotenv import load_dotenv
load_dotenv()

def create_llm(deployment_name):
    return AzureChatOpenAI(
        api_key=os.getenv("API_KEY"),
        api_version=os.getenv("AZURE_API_VERSION"),
        azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        deployment_name=deployment_name,
        temperature=0.2,
        max_tokens=800
    )

# model map (future-proof)
llm_map = {
    "fast": create_llm("gpt-4.1"),
    # later: "smart": create_llm("gpt-5.1")
}

#### Define Prompts for each step as we did earlier


In [2]:
#Step 1 — Planning
decompose_prompt = ChatPromptTemplate.from_messages([
    ("system", "Break the question into clear logical steps."),
    ("human", "{input}")
])

In [3]:
#Step 2 — Initial reasoning
reason_prompt = ChatPromptTemplate.from_messages([
    ("system", "Solve the problem step by step using the plan."),
    ("human", "Question: {input}\nPlan: {plan}")
])

In [4]:
#Step 3 — Critique (This forces the model to think like a reviewer)
critique_prompt = ChatPromptTemplate.from_messages([
    ("system", "Critically evaluate the solution. Identify mistakes, gaps, or unclear reasoning."),
    ("human", "Question: {input}\nSolution: {solution}")
])

In [5]:
#Step 4 - Improve answer
improve_prompt = ChatPromptTemplate.from_messages([
    ("system", "Improve the answer using the critique. Provide a better final answer."),
    ("human", "Question: {input}\nSolution: {solution}\nCritique: {critique}")
])

In [6]:
#Build pipeline
def self_correcting_reasoning(question: str, model_choice="fast", debug=False):
    llm = llm_map[model_choice]

    # Step 1: Plan
    plan = (decompose_prompt | llm).invoke({"input": question}).content

    # Step 2: Initial solution
    solution = (reason_prompt | llm).invoke({
        "input": question,
        "plan": plan
    }).content

    # Step 3: Critique
    critique = (critique_prompt | llm).invoke({
        "input": question,
        "solution": solution
    }).content

    # Step 4: Improved answer
    final = (improve_prompt | llm).invoke({
        "input": question,
        "solution": solution,
        "critique": critique
    }).content

    if debug:
        print("\n PLAN:\n", plan)
        print("\n INITIAL SOLUTION:\n", solution)
        print("\n CRITIQUE:\n", critique)
        print("\n FINAL ANSWER:\n", final)

    return {
        "plan": plan,
        "initial_solution": solution,
        "critique": critique,
        "final_answer": final
    }

In [7]:
#Test it
result = self_correcting_reasoning(
    "Why do planets orbit the sun?",
    debug=True
)

print("\nFINAL:\n", result["final_answer"])


 PLAN:
 Certainly! Let’s break down the question “Why do planets orbit the sun?” into clear logical steps:

---

**1. The Sun’s Gravity**

- The sun is extremely massive—over 99% of the mass in our solar system.
- All objects with mass exert a gravitational pull on other objects.
- The sun’s gravity pulls on the planets, attracting them toward itself.

---

**2. The Planets’ Motion**

- Planets are not stationary; they move sideways (tangentially) at high speeds.
- If a planet were at rest near the sun, it would fall straight in due to gravity.
- However, because planets are moving, their inertia (tendency to move in a straight line) counteracts the pull of gravity.

---

**3. The Balance: Orbit Formation**

- The planet’s forward motion tries to carry it away from the sun.
- The sun’s gravity tries to pull the planet inward.
- The result is a balance: the planet continuously “falls” toward the sun but also moves forward, causing it to travel in a curved path around the sun—a stable o

In [11]:
#Summary
#Without critique
   #LLM → answer (maybe wrong)
#With critique
#LLM → answer → self-review → improved answer

#### Improving to use critique loops, guardrails & structured responses

In [8]:
#Prompts (structured + guardrails)
#Planning
plan_prompt = ChatPromptTemplate.from_messages([
    ("system", "Break the question into clear logical steps."),
    ("human", "{input}")
])

In [9]:
#Initial solution (with Guardrails)
reason_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Solve step by step.\n"
     "Follow these rules:\n"
     "- Be accurate\n"
     "- Do not make up facts\n"
     "- If unsure, say 'I am not sure'\n"
     "- Keep reasoning clear"),
    ("human", "Question: {input}\nPlan: {plan}")
])

In [10]:
#Structured critique (to have output structured & predictable)
critique_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a strict reviewer.\n"
     "Analyze the solution and respond in this EXACT format:\n\n"
     "Mistakes:\n- ...\n\n"
     "Missing Information:\n- ...\n\n"
     "Improvements:\n- ...\n\n"
     "If no issues, say 'None' under each section."),
    ("human", "Question: {input}\nSolution: {solution}")
])

In [11]:
#Improve with guardrails
improve_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Improve the answer using the critique.\n"
     "Rules:\n"
     "- Fix all mistakes\n"
     "- Add missing information\n"
     "- Keep it concise and clear\n"
     "- Do not introduce new errors"),
    ("human",
     "Question: {input}\n"
     "Previous Answer: {solution}\n"
     "Critique: {critique}")
])

In [12]:
#Multi-loop self-correcting function
def self_correcting_reasoning(
    question: str,
    model_choice="fast",
    critique_loops=2,
    debug=False
):
    llm = llm_map[model_choice]

    # Step 1: Plan
    plan = (plan_prompt | llm).invoke({"input": question}).content

    # Step 2: Initial solution
    solution = (reason_prompt | llm).invoke({
        "input": question,
        "plan": plan
    }).content

    history = []

    #Step 3: Multiple critique + improve loops
    for i in range(critique_loops):
        critique = (critique_prompt | llm).invoke({
            "input": question,
            "solution": solution
        }).content

        improved_solution = (improve_prompt | llm).invoke({
            "input": question,
            "solution": solution,
            "critique": critique
        }).content

        # store iteration history
        history.append({
            "iteration": i + 1,
            "critique": critique,
            "improved_solution": improved_solution
        })

        solution = improved_solution  # update for next loop

    if debug:
        print("\n PLAN:\n", plan)
        print("\n FINAL SOLUTION AFTER LOOPS:\n", solution)

        for h in history:
            print(f"\n ITERATION {h['iteration']}")
            print("CRITIQUE:\n", h["critique"])
            print("IMPROVED:\n", h["improved_solution"])

    return {
        "plan": plan,
        "final_answer": solution,
        "iterations": history
    }

In [13]:
result = self_correcting_reasoning(
    "Why do planets orbit the sun?",
    critique_loops=2,
    debug=True
)

print("\nFINAL ANSWER:\n", result["final_answer"])


 PLAN:
 Certainly! Let’s break down the question “Why do planets orbit the sun?” into clear logical steps:

**1. The Sun’s Mass and Gravity**
- The Sun is much more massive than any planet in the solar system.
- All objects with mass exert a gravitational force, and the Sun’s gravity is the dominant force in the solar system.

**2. Gravitational Attraction**
- The Sun’s gravity pulls planets toward it.
- If gravity were the only force, planets would fall straight into the Sun.

**3. Planetary Motion and Inertia**
- Planets are also moving sideways (tangentially) at high speeds.
- According to Newton’s First Law (inertia), an object in motion stays in motion in a straight line unless acted upon by a force.

**4. The Balance: Orbit Formation**
- The planet’s forward motion tries to carry it away from the Sun.
- The Sun’s gravity pulls the planet inward.
- The balance between these two effects causes the planet to move in a curved path around the Sun—an orbit.

**5. Result: Elliptical Or

In [ ]:
#Note
#Cost - increases (multiple calls)
#Latency - slower
#Quality	- much better